# Семинар 10. Python. Работа с текстами

## Мотивация

Данные исследователя почти всегда сначала приходят текстом: лог обучения, выгрузка
из чужой системы, csv от коллеги, ответ API. Пока текст не превращён в числа и
таблицы, анализировать нечего — а на этом пути ровно три типовые боли.

1. **Файл открывается «кракозябрами»** или падает с `UnicodeDecodeError`. Данные
   целы, но прочитаны не тем правилом — надо понимать, что такое кодировка.
2. **Нужные файлы разбросаны** по каталогам экспериментов: `runs/run-01/`,
   `runs/run-02/`, архив. Собрать их руками нельзя — нужна маска.
3. **Из строки лога надо достать значения**: время, номер эпохи, loss. Пока формат
   строго колоночный, хватает `split`; как только он «плавает», нужен более
   выразительный язык описания — регулярные выражения.

Сегодня разбираем эти три инструмента: кодировки (`bytes` и `str`), поиск файлов
по маске (`glob`, `pathlib`) и регулярные выражения (`re`). Домашка — парсер
текстового лога на регулярных выражениях, так что всё пригодится сразу.

В семинаре 3 похожие задачи решались командами оболочки: `grep`, `find`, `wc`.
Разница не в том, что «одно лучше другого»: командная строка удобна для быстрого
взгляда на файл, а Python сразу отдаёт результат объектом, который можно
посчитать, положить в таблицу и построить график.

Всё, что создаём на семинаре, пишем в домашний каталог — `~/seminar-10/`.
В `/tmp` работать нельзя: он вычищается при перезагрузке, а результат нужен и
после занятия.

## 1. Байты и кодировки

В файле и в сети нет «текста» — есть последовательность байт. **Кодировка** —
правило, по которому символ превращается в байты и обратно. В Python эти две
сущности разделены: `str` — последовательность символов, `bytes` —
последовательность байт. Перевод делают `encode` (символы → байты) и
`decode` (байты → символы).

Весь путь целиком — на схеме. Верхняя половина понадобится прямо сейчас, нижняя
— через несколько ячеек, когда дойдём до кракозябр.

Символ, кодовая точка, байты и происхождение кракозябры

Новые файлы всегда пишем в UTF-8: это умолчание всего современного стека —
интернета, git, Python. `cp1251` и `koi8-r` нужны только чтобы **прочитать**
старую выгрузку, писать в них ничего нового не надо.

In [ ]:
text = "Привет, мир"
data = text.encode("utf-8")   # str → bytes: символы записываются байтами
print(len(text), len(data))   # символов и байт — разное количество
print(data)                   # латиница видна как есть, кириллица — как \xd0\x9f

Символов 11, а байт 20: в UTF-8 кириллическая буква занимает два байта,
а запятая и пробел — по одному. Поэтому длина строки и размер файла — разные
величины.

Длина закодирована в самих байтах — это видно по их старшим битам:

```
0xxxxxxx                     1 байт  — ASCII (латиница, цифры, знаки)
110xxxxx 10xxxxxx            2 байта — кириллица
1110xxxx 10xxxxxx 10xxxxxx   3 байта — иероглифы, часть символов
11110xxx 10xxxxxx 10xxxxxx 10xxxxxx   4 байта — эмодзи, редкие письменности
```

Первый байт объявляет длину (`0`, `110`, `1110`, `11110` — больше четырёх байт
в UTF-8 не бывает), продолжающие обязаны начинаться с `10`. Запомните эту
структуру — через несколько ячеек она объяснит, почему одна ошибка кодировки
тихая, а другая шумная.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

UTF-8 придумали Кен Томпсон и Роб Пайк в сентябре 1992-го: набросали схему
на салфетке в придорожной закусочной в Нью-Джерси и за выходные вкрутили её в
Plan 9. Главная хитрость — как раз структура старших битов: по любому байту
видно, начало это символа или продолжение. Отсюда два практических следствия.
Можно прыгнуть на середину огромного файла и сразу найти границу символа, не
читая его с начала. И потерянный при передаче кусок портит несколько букв, а не
весь хвост файла — декодер снова синхронизируется на первом же ведущем байте.
У UTF-16 такого свойства нет, и это одна из причин, почему победил UTF-8.

</details>

Кодировка UTF-8 не единственная. Старые выгрузки часто приходят в
однобайтовой `cp1251` (Windows-кириллица) или `koi8-r`.

In [ ]:
data_1251 = text.encode("cp1251")  # тот же текст, но однобайтовой кодировкой
print(len(data_1251), data_1251)   # 11 байт вместо 20: по байту на любую букву
print(text.encode("koi8-r"))       # длина та же, а байты другие

Одному и тому же тексту соответствуют **разные** байты: `cp1251` и `koi8-r` дали
одинаковую длину (обе однобайтовые), но разные значения байт — буквы в них
расставлены в разном порядке. Значит, байты без указания кодировки прочитать
нельзя — можно только угадать, и по одной длине два таких файла не различить
(это пригодится в задаче H3).

Если правило чтения то же, что и правило записи, текст возвращается целым.

In [ ]:
print(data_1251.decode("cp1251"))  # прочитали тем же правилом — текст как был

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Порядок букв в KOI8-R выбран не по алфавиту, а так, чтобы при потере
восьмого бита (а линии связи в 1970-х теряли его регулярно) кириллица
превращалась в латинскую транслитерацию: «Привет» становится «pRIWET» —
некрасиво, но прочитать можно. В cp1251 буквы идут по алфавиту, и при той же
потере получается неразборчивый мусор.

А слово «кракозябры» — наше, местное. В англоязычной литературе это явление
называют японским словом mojibake (文字化け, «превращение символов»): в Японии
кодировок было больше, и с проблемой там столкнулись раньше и массовее.

</details>

Теперь посмотрим, что бывает при неверной догадке о кодировке.

In [ ]:
broken = text.encode("utf-8").decode("cp1251")  # записали в utf-8, читаем как cp1251
print(broken)                                   # 20 байт стали 20 «не теми» символами

Это и есть «кракозябры» (mojibake): файл не испорчен, верные байты прочитаны
неверным правилом — ровно нижняя половина схемы в начале раздела. Такая порча
обратима: раз каждому байту нашёлся символ, ни один байт не потерялся — обратная
пара `encode`/`decode` (в том же порядке кодировок) вернёт исходный текст. Это и
есть задача B2 из `tasks.md`.

Обратная ошибка выглядит иначе.

In [ ]:
try:
    data_1251.decode("utf-8")            # байты cp1251 не складываются в utf-8
except UnicodeDecodeError as exc:
    print(exc)                           # видно, на каком байте разбор встал

#### ❓ **Вопрос**: Почему UTF-8-байты, прочитанные как cp1251, дали мусор, а cp1251-байты, прочитанные как UTF-8, — исключение?

<details>

<summary><strong>Ответ</strong></summary>

`cp1251` однобайтовая: символ сопоставлен почти каждому из 256 байт (единственная дыра — байт `0x98`: он в таблице не определён, и декодировать его как cp1251 не выйдет), а структурных требований к последовательности нет. Поэтому почти любые байты «читаются» — просто не тем текстом.

`UTF-8` многобайтовая, и в ней у байт есть структура: первый байт многобайтового символа объявляет длину, продолжающие байты обязаны начинаться с `10`. Байты cp1251 этому не подчиняются, и `decode` сообщает об ошибке вместо тихой порчи данных.

</details>

Если данные важнее полноты, `decode` можно попросить не падать. Но это
осознанная потеря: аргумент `errors` подменяет или выбрасывает то, что не
разобралось.

In [ ]:
print(data_1251.decode("utf-8", errors="replace"))  # непонятное → символ-заменитель
print(data_1251.decode("utf-8", errors="ignore"))   # непонятное просто выброшено

`errors="replace"` ставит символ-заменитель U+FFFD (в Python он записывается как
`"\ufffd"`, а в выводе выше это ромбики с вопросительным знаком) — по одному на
**каждую неразобранную последовательность**, а не на каждый байт: у нас байты
cp1251 плохи поодиночке, а вот обрезанный UTF-8-символ `b"\xe2\x82"` даст один
ромбик на два байта. `errors="ignore"` молча выбрасывает байты. Для разведки
годится, для боевого разбора — нет: испорченные строки должны быть видны, а не
исчезать.

Теперь файлы. Пути будем собирать через `pathlib`: `Path("...")` — это объект
пути, оператор `/` приклеивает к нему следующий кусок (`work / "report.txt"`), а
`read_text`/`write_text`/`read_bytes` читают и пишут файл целиком одной строкой.
Подробнее про `Path` — в разделе 3, здесь достаточно этих трёх фактов.

Рабочий каталог заводим в домашнем — `~/seminar-10/demo`.

In [ ]:
from pathlib import Path
work = Path.home() / "seminar-10" / "demo"  # домашний каталог, а не /tmp
work.mkdir(parents=True, exist_ok=True)     # каталог мог остаться с прошлого запуска
print(work)                                 # сюда лягут все файлы раздела

У `open`, `Path.read_text` и `Path.write_text` есть параметр `encoding`.
**Указывать его нужно явно.** Без него Python берёт кодировку локали — а она у
каждой машины своя.

In [ ]:
import locale
print(locale.getpreferredencoding(False))  # это правило возьмёт open() без encoding

На Linux в выводе обычно UTF-8, а на Windows — ANSI-кодовая страница локали
(в русской это cp1251, в западноевропейской — cp1252). Один и тот же скрипт
ведёт себя на разных машинах по-разному, и это ловится не сразу.

Заведём файл в чужой кодировке — как будто его прислали из старой системы.

In [ ]:
report = work / "report-1251.txt"                         # / приклеивает имя к пути
report.write_text("Модель обучена\n", encoding="cp1251")  # пишем не в utf-8 намеренно
print("записан:", report.name)                            # файл лежит в ~/seminar-10/demo

Файл на диске — это байты. Посмотрим и на них, и на текст.

In [ ]:
print(report.read_bytes())                          # b'\xcc...' — однобайтовая кириллица
print(report.read_text(encoding="cp1251"), end="")  # правило чтения совпало с записью

#### ❓ **Вопрос**: Файл записан в cp1251. Что произойдёт на Linux при чтении `report.read_text()` без `encoding`?

<details>

<summary><strong>Ответ</strong></summary>

Возьмётся кодировка локали — та, которую напечатала ячейка с `locale.getpreferredencoding` (на этой машине UTF-8; на другой в выводе может быть и `ANSI_X3.4-1968`). Байты cp1251 не раскладываются ни в UTF-8, ни в ASCII, поэтому будет `UnicodeDecodeError`, как в примере выше. А вот в русской Windows та же строка отработала бы без ошибки — результат зависит от машины, отсюда и правило указывать `encoding` явно.

</details>

Отдельная ловушка — **BOM** (byte order mark): три служебных байта в начале
файла, которыми Excel помечает UTF-8. Кодировка `utf-8-sig` их пишет и снимает,
а обычная `utf-8` о них не знает и отдаёт как невидимый символ.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

BOM придумали не для UTF-8. `U+FEFF` — «метка порядка байт»: в UTF-16 символ
занимает два байта, и записать их можно в двух порядках, а метка в начале файла
говорит, в каком именно. В UTF-8 порядок байт зафиксирован структурой самих
байт, и метка там не нужна вообще ни для чего — но windows-программы стали
писать её как «подпись»: мол, файл точно в UTF-8. Отсюда и `sig` в названии
кодировки `utf-8-sig` — signature. Стандарт Unicode такую подпись не
рекомендует, а Excel всё равно её ставит, и раз в семестр кто-нибудь на этом
теряет полдня.

</details>

Увидеть невидимое помогает `repr`: он печатает строку так, как её записали бы в
коде — с кавычками и escape-последовательностями (`'\ufeffepoch'`, `'0.71\n'`).
Обычный `print` тех же символов просто не покажет. В следующих ячейках ещё
встретится `split(",")` — он режет строку по запятой и возвращает список кусков;
подробно строковые методы разбираем в разделе 2.

In [ ]:
bom_file = work / "excel.csv"                                        # csv «как из Excel»
bom_file.write_text("epoch,loss\n1,0.7100\n", encoding="utf-8-sig")  # -sig допишет BOM
print(bom_file.read_bytes()[:3])                                     # вот они, три байта

Теперь прочитаем имя первой колонки двумя способами и сравним.

In [ ]:
print(repr(bom_file.read_text(encoding="utf-8").split(",")[0]))      # BOM прилип к имени
print(repr(bom_file.read_text(encoding="utf-8-sig").split(",")[0]))  # тут его сняли при чтении

#### ❓ **Вопрос**: В имени колонки застрял невидимый символ из BOM. Уберёт ли его привычный `strip()`?

<details>

<summary><strong>Ответ</strong></summary>

Нет. `strip()` без аргументов снимает только пробельные символы, а `\ufeff` к ним не относится: `'\ufeff'.isspace()` — это `False`. Поэтому `'\ufeffepoch'.strip()` вернёт строку без изменений, и колонка по-прежнему не найдётся.

Сработал бы явный `strip('\ufeff')` (набор символов задан руками) — но это лечение симптома. Правильно читать файл с `encoding="utf-8-sig"`: тогда BOM снимается при декодировании и в тексте его просто нет.

</details>

## 2. Операции над строками

Примеры раздела — строки из того самого лога `train.log`, парсер которого
соберём в разделе 5. Пока формат строки строгий, этих методов достаточно: ровно
так, на `split` и `partition`, решается задача B4.

Строки в Python **неизменяемы**: методы не правят строку на месте, а возвращают
новую. Поэтому разбор строки — это цепочка «отрезали → разделили → преобразовали».

In [ ]:
line = "  2026-11-03 12:04:12 INFO epoch=1 loss=0.7100  \n"
print(repr(line.strip()))  # repr покажет, что пробелы и \n по краям исчезли
print(line.split())        # без аргументов режет по любым пробелам сразу

`strip()` без аргументов снимает пробельные символы с обоих концов, включая
перевод строки, — это первое, что делают со строкой, прочитанной из файла.
`split()` без аргументов режет по любым пробелам и выбрасывает пустые куски.
Для csv так делать нельзя: там важны и разделитель, и пустые поля.

In [ ]:
print("1,,0.7100".split(","))  # пустое поле между запятыми сохранилось
print("a b  c".split())        # любые пробелы, пустых кусков нет
print("a b  c".split(" "))     # ровно один пробел — между двумя даёт пустой кусок

`split(",")` сохранил пустое поле между запятыми, `split(" ")` — пустой кусок
между двумя пробелами. Разделять «по одному конкретному символу» и «по любым
пробелам» — разные задачи.

Пару «ключ = значение» удобнее разбирать через `partition`: он всегда возвращает
три части и не падает, если разделителя нет.

In [ ]:
key, sep, value = "loss=0.7100".partition("=")  # до разделителя, сам он, после
print(repr(key), repr(value))
print("no-sign".partition("="))                 # разделителя нет — два куска пустые

Снять фиксированный префикс или суффикс нужно методами `removeprefix` и
`removesuffix` (Python 3.9+). Соблазн сделать это через `strip` — типичная и очень
живучая ошибка.

In [ ]:
name = "epoch_metrics.csv"
print(name.strip(".csv"))         # съело лишнее: strip получает НАБОР символов
print(name.removesuffix(".csv"))  # а этот метод снимает именно подстроку

#### ❓ **Вопрос**: Почему `"epoch_metrics.csv".strip(".csv")` вернул `epoch_metri`, а не `epoch_metrics`?

<details>

<summary><strong>Ответ</strong></summary>

`strip` принимает **набор символов**, а не подстроку. Набор здесь — `{'.', 'c', 's', 'v'}`, и с правого конца снимаются все символы из этого набора подряд: `v`, `s`, `c`, `.`, `s`, `c`. Остановка происходит на `i`, которого в наборе нет. Убрать именно суффикс умеет `removesuffix`.

</details>

Проверить префикс или суффикс, ничего не отрезая, умеют `startswith` и
`endswith`. Так отбирают файлы по расширению и отсеивают строки, не похожие на
начало записи лога.

In [ ]:
print(name.endswith(".csv"), name.startswith("epoch"))
print(line.startswith("2026-11-03"))          # False: впереди пробелы из файла
print(line.strip().startswith("2026-11-03"))  # сняли пробелы — и проверка сошлась

Второй вызов вернул `False`: в строке из файла впереди пробелы, и «начинается
с даты» ломается ровно на них — поэтому `strip()` делают первым делом. Оба
метода принимают и кортеж вариантов: `name.endswith((".csv", ".tsv"))`.

Важно помнить их границу: они сравнивают с **фиксированной** подстрокой. Как
только префикс переменный («строка начинается с любой даты»), нужен уже
`re.match` — им и займёмся в разделе 4.

Ещё две рабочие лошадки — `lower`/`upper` (регистр) и `replace` (заменить все
вхождения подстроки). Вместе с `removesuffix` они дают нормализацию имени: файлы,
названные по-разному, сводятся к одной форме и становятся сравнимы.

In [ ]:
raw_name = "RUN-05_2026-11-05.CSV"
print(raw_name.lower())  # регистр в именах файлов бывает какой угодно
print(raw_name.lower().removesuffix(".csv").replace("-", "_"))  # одна общая форма

Обратная операция — сборка строки. `join` склеивает готовый список: строка, у
которой вызвали метод, становится разделителем.

In [ ]:
fields = ["2026-11-03", "1", "0.7100"]
print(",".join(fields))  # получилась строка csv; разделитель — та строка, что слева

Подставить значения в шаблон удобнее f-строкой. Формат пишут после двоеточия
внутри фигурных скобок: `{n:03d}` — целое, дополненное нулями до трёх знаков,
`{x:.2f}` — число с двумя знаками после точки, `{x:.4f}` — с четырьмя
(пригодится в H4).

In [ ]:
loss = 0.71
print(f"epoch={1:03d} loss={loss:.2f}")  # 001 и 0.71 — ширина и точность заданы форматом

#### ❓ **Вопрос**: Строку `"loss=0.7100\n"` прочитали из файла и сразу применили `partition("=")`. Что окажется в значении и почему сначала нужен `strip()`?

<details>

<summary><strong>Ответ</strong></summary>

В значении окажется `'0.7100\n'` — вместе с переводом строки, потому что `partition` режет строку по разделителю и ничего не отбрасывает. Дальше `float('0.7100\n')` ещё сработает, а вот сравнение с `'0.7100'` или запись в csv уже дадут неожиданный результат. `strip()` снимает пробельные символы с концов, включая `\n`, — это видно по первой ячейке раздела, где `repr` показал строку до и после.

</details>

## 3. Поиск файлов по маске: glob

Эксперимент оставляет после себя дерево каталогов: `runs/run-01/`, `runs/run-02/`,
архив со старыми запусками. Задача «взять все `metrics.csv`» руками не решается —
нужна **маска имени** (glob-шаблон):

- `*` — любое число любых символов, кроме разделителя каталогов;
- `?` — ровно один символ;
- `[0-9]` — один символ из класса;
- `**` — любое число уровней вложенности (только с `recursive=True`).

Сначала соберём учебное дерево.

In [ ]:
runs = work / "runs"                                    # ~/seminar-10/demo/runs
for rel in ["run-01/metrics.csv", "run-02/metrics.csv",
            "run-02/notes.txt", "archive/run-03/metrics.csv"]:
    path = runs / rel
    path.parent.mkdir(parents=True, exist_ok=True)      # создаст и промежуточные каталоги
    path.write_text("epoch,loss\n", encoding="utf-8")   # содержимое сейчас неважно

Дерево готово. Посмотрим, что лежит на верхнем уровне: `iterdir` — это
«показать содержимое каталога», один уровень, без всякой маски.

In [ ]:
print(sorted(p.name for p in runs.iterdir()))  # два запуска и каталог архива

Теперь то же дерево, но через маску.

In [ ]:
import glob
print(glob.glob(str(runs / "*" / "metrics.csv")))  # звёздочка = ровно один уровень

Один `*` соответствует ровно одному уровню вложенности, поэтому файл из
`archive/run-03/` не нашёлся. Рекурсивный поиск включает `**` вместе с
`recursive=True`.

In [ ]:
found = glob.glob(str(runs / "**" / "metrics.csv"), recursive=True)  # ** = любая глубина
print(sorted(found))  # теперь виден и файл из архива

То же самое `pathlib` делает методами `glob` и `rglob` (`rglob` — рекурсивный
вариант). Результат — объекты `Path`, а не строки: у них сразу есть `.name`,
`.parent`, `.suffix`.

Что выбирать: `glob.glob` удобен в однострочниках и когда нужны просто строки
путей (например, сразу отдать их в другую функцию); `pathlib` выигрывает, как
только с найденными путями ещё работать — резать имя, брать каталог, склеивать
новый путь. Устаревшим `glob` при этом не является.

In [ ]:
for path in sorted(runs.rglob("metrics.csv")):        # rglob = glob с ** в начале
    print(path.relative_to(runs), path.parent.name)   # путь от корня дерева и имя запуска

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Слово glob — из ранних Unix: в 1970-х раскрывать маски умела не оболочка, а
отдельная программа `/etc/glob`, которую shell запускал, встретив звёздочку в
аргументах. Потом раскрытие переехало внутрь оболочки, программа исчезла, а имя
осталось — и в названии модуля Python в том числе. Побочный эффект той истории
живёт до сих пор: маску раскрывает оболочка, а не программа, поэтому
`grep '*.log'` в кавычках и `grep *.log` без кавычек ведут себя по-разному.

</details>

Остальные шаблоны из шапки раздела работают так же, как в оболочке: `[0-9]` —
один символ из диапазона, `?` — ровно один любой символ.

In [ ]:
print(sorted(p.parent.name for p in runs.glob("run-0[1-9]/metrics.csv")))  # класс символов
print(sorted(runs.glob("run-?/metrics.csv")))                              # пусто — почему?
print(sorted(p.parent.name for p in runs.glob("run-??/metrics.csv")))      # а так нашлось

Маска `run-?` не нашла ничего: `?` — это **ровно один** символ, а в имени
`run-01` после дефиса их два, поэтому подходит `run-??`. Класс `[1-9]` полезен,
когда надо взять подмножество запусков, не перечисляя их имена.

#### ❓ **Вопрос**: Почему результат `glob` стоит оборачивать в `sorted`?

<details>

<summary><strong>Ответ</strong></summary>

Порядок выдачи не определён: он зависит от порядка записей в файловой системе. Пока порядок не зафиксирован, один и тот же скрипт на двух машинах даст разный порядок строк в отчёте, а diff покажет ложные изменения. `sorted` делает результат воспроизводимым.

</details>

Скрытые файлы (имя начинается с точки) два инструмента обрабатывают
по-разному. Положим такой файл в `runs/`.

In [ ]:
(runs / ".hidden.csv").write_text("epoch,loss\n", encoding="utf-8")  # имя с точки
print(sorted(p.name for p in runs.iterdir()))  # iterdir скрытый файл показывает

И запросим обоими инструментами одну и ту же маску.

In [ ]:
print(glob.glob(str(runs / "*.csv")))        # модуль glob скрытые файлы не показывает
print([p.name for p in runs.glob("*.csv")])  # а pathlib показывает

#### ❓ **Вопрос**: Маска одна и та же, а `glob.glob` вернул пустой список, тогда как `Path.glob` нашёл `.hidden.csv`. Кто из них ошибается?

<details>

<summary><strong>Ответ</strong></summary>

Никто: это разное поведение по документации. `glob.glob` повторяет правило оболочки — `*` не совпадает с точкой в начале имени, скрытые файлы запрашивают явной маской `.*.csv`. У `pathlib` такого правила нет, и `Path.glob` возвращает скрытые файлы наравне с обычными.

Практический вывод: переписывая код с одного инструмента на другой, результат можно молча изменить. Если скрытые файлы для задачи не нужны — отфильтруйте их сами, а не полагайтесь на умолчание.

</details>

## 4. Регулярные выражения

`split` хорош, пока формат строгий. Как только в строке появляется переменная
часть — необязательное поле, разное число пробелов, посторонние строки, — нужен
язык, описывающий **множество** подходящих строк. Это регулярные выражения; в
оболочке их использует `grep` (семинар 3), в Python — модуль `re`.

Сразу предупреждение о путанице: `*` и `?` есть и в маске glob из прошлого
раздела, и в регулярке, но значат совершенно разное.

Одна и та же запись в маске glob и в регулярке


Основные элементы шаблона:

| Запись | Что значит |
|---|---|
| `\d` `\w` `\s` | цифра, буква/цифра/подчёркивание, пробельный символ |
| `.` | любой символ, кроме перевода строки |
| `+` `*` `?` | одно и больше, ноль и больше, ноль или одно |
| `{2,4}` | от двух до четырёх повторов |
| `[A-Z]` | один символ из класса |
| `^` `$` | начало и конец строки |
| `(...)` | группа: то, что нужно достать |
| `(?:...)` | группировка без захвата — в результат не попадает |
| `a\|b` | альтернатива: подходит либо `a`, либо `b` |
| `\S` `\W` `\D` | отрицание класса: не пробел, не буква/цифра/подчёркивание, не цифра |
| `\b` | граница слова: место, где `\w` встречается с не-`\w` (начало или конец слова) |
| `\.` `\+` `\\` | экранирование: сам символ, а не его особый смысл |

Точка, плюс, скобки, звёздочка — метасимволы. Чтобы найти именно точку, её
экранируют обратной косой: `\.` — это точка, а `.` — любой символ. Поэтому в
первом же примере ниже стоит `loss=(\d+\.\d+)`, а не `loss=(\d+.\d+)`: второй
шаблон совпал бы и с `loss=1x5`.

Шаблоны пишут **r-строками**: в обычной строке Python `\b` — это забой, а не
граница слова, и шаблон незаметно меняет смысл.

In [ ]:
import re
line = "2026-11-03 12:04:12 INFO epoch=1 loss=0.7100 acc=0.6210"
found = re.search(r"loss=(\d+\.\d+)", line)  # \. — именно точка, а не «любой символ»
print(found)                                 # объект Match: где нашлось и что именно

`search` вернул не строку, а объект `Match` — из него значения достают методом
`group`. Если совпадения нет, возвращается `None`, и проверять это обязательно:
`None.group(...)` — самая частая ошибка при разборе логов.

In [ ]:
print(found.group(0))  # всё совпадение целиком
print(found.group(1))  # только первая группа в скобках — то, что нужно достать

У `re` три похожих функции поиска, и разница между ними — источник половины
ошибок новичков. Отличаются они только тем, где шаблону разрешено совпасть.

search ищет везде, match — только с начала, fullmatch — строку целиком

In [ ]:
print(re.search(r"epoch=(\d+)", line).group(1))  # нашёл в середине строки
print(re.match(r"epoch=(\d+)", line))            # None: строка начинается с даты
print(re.fullmatch(r"\d+", "42"))                # шаблон покрыл строку целиком

#### ❓ **Вопрос**: `re.search` нашёл `epoch=1`, а `re.match` вернул `None` для того же шаблона и той же строки. Почему?

<details>

<summary><strong>Ответ</strong></summary>

`re.match` пытается совпасть **с начала строки**, а строка начинается с даты `2026-11-03`, а не с `epoch=`. `re.search` ищет совпадение в любом месте. Третья функция, `re.fullmatch`, требует, чтобы шаблон покрыл всю строку целиком, — она удобна для проверки формата.

</details>

В таблице выше остались невостребованными счётчик повторов `{n,m}` и класс
`[A-Z]`. В примерах ниже впервые появится `findall` — он возвращает **все** совпадения
списком: пока в шаблоне нет скобок-групп, это список найденных строк. Что
меняется с группами, разберём через несколько ячеек, вместе с `finditer`.
Оба элемента нужны, когда важна форма поля: дата — это ровно четыре цифры, дефис
и две пары цифр, а уровень лога — заглавные латинские буквы.

In [ ]:
head = "2026-11-03 12:04:12 WARNING nan detected in batch 17"
print(re.search(r"\d{4}-\d{2}-\d{2}", head).group(0))  # ровно 4-2-2 цифры — это дата
print(re.findall(r"[A-Z]{2,}", head))                  # два и больше заглавных подряд

Заглавная буква инвертирует класс: `\S` — «любой непробельный символ», `\D` —
«любая не-цифра». А `\b` — граница слова: место, где `\w`-символ соседствует с
не-`\w` или с краем строки. Сама она ничего не съедает, а лишь проверяет условие —
и с какой стороны шаблона её поставить, важно: `ERROR\b` требует границу справа и
отсеивает `ERRORS`, а `\bERROR` проверял бы границу слева и нашёл бы оба
вхождения.

In [ ]:
print(re.findall(r"\S+", "epoch=1  loss=0.7100"))         # куски без пробелов
print(re.findall(r"ERROR", "ERRORS: ERROR at line 3"))    # нашёл и внутри ERRORS
print(re.findall(r"ERROR\b", "ERRORS: ERROR at line 3"))  # граница справа отсекла ERRORS

Ещё две записи из таблицы — альтернатива `a|b` и не-захватывающая группа
`(?:...)`. Скобки в регулярке делают две разные вещи сразу: группируют и
захватывают. Когда нужна только группировка, пишут `(?:...)`.

In [ ]:
print(re.findall(r"(?:INFO|WARNING|ERROR)", head))      # (?: ) только перечисляет варианты
print(re.findall(r"(INFO|WARNING|ERROR) (\w+)", head))  # обычные скобки — уже захват

#### ❓ **Вопрос**: Почему первый `findall` вернул само совпадение, а второй — кортеж?

<details>

<summary><strong>Ответ</strong></summary>

Если в шаблоне нет захватывающих групп, `findall` отдаёт текст самого совпадения (`'WARNING'`, а не всю строку): `(?:...)` только перечисляет варианты и в результат не попадает. Как только появились обычные скобки, `findall` переключается на содержимое групп — по кортежу на совпадение.

Отсюда практическое правило: скобки, поставленные «просто чтобы сгруппировать», молча меняют форму результата. Не нужен захват — пишите `(?:...)`.

</details>

Скобки вокруг альтернативы нужны ещё и потому, что `|` — самый «слабый» оператор
в шаблоне: он делит **всё выражение** пополам.

In [ ]:
print(re.findall(r"^INFO|WARNING", head))      # читается как (^INFO) или (WARNING где угодно)
print(re.findall(r"^(?:INFO|WARNING)", head))  # а так — «в начале строки одно из двух»

Когда групп несколько, нумеровать их неудобно: добавили поле в середину — и
все номера уехали. Группам дают имена через `(?P<имя>...)`. Как шаблон
раскладывается на куски и что из них попадает в группы — на схеме.

Разбор шаблона по частям и содержимое именованных групп

In [ ]:
pattern = re.compile(r"(?P<time>\S+ \S+) (?P<level>\w+) epoch=(?P<epoch>\d+)")
found = pattern.search(line)  # у скомпилированного шаблона те же методы

Достать группы теперь можно по имени, а не по номеру.

In [ ]:
print(found.group("level"), found.group("epoch"))  # по одной группе
print(found.groupdict())                           # все именованные сразу, словарём

`re.compile` заранее разбирает шаблон. Главная польза не в скорости, а в том,
что шаблон получает имя и перестаёт дублироваться по коду.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Разбирать строку шаблона на каждом вызове `re.search` Python и так не станет:
модуль `re` держит внутренний кеш последних скомпилированных шаблонов (сейчас —
512 штук, общий на процесс). Так что `compile` в цикле почти ничего не
ускоряет — если только шаблонов у вас не больше, чем помещается в кеш, и они не
вытесняют друг друга. Но когда шаблон длинный, `compile` вытаскивает его из
середины кода наверх и даёт ему осмысленное имя, а это важнее.

</details>

Чтобы достать все совпадения, а не первое, есть `findall` и `finditer`.

In [ ]:
metrics = "loss=0.7100 acc=0.6210 loss=0.4213"
print(re.findall(r"(\w+)=([\d.]+)", metrics))  # две группы → список кортежей
print(re.findall(r"\w+=", metrics))            # групп нет → список самих совпадений

`findall` возвращает список: без групп — из самих совпадений, а с группами —
из кортежей, по одному кортежу **на совпадение** и по элементу в кортеже на
каждую группу (если группа одна, кортежей нет — сразу её содержимое). `finditer`
вместо этого отдаёт объекты `Match`, у которых есть и группы, и позиции — он
нужен, когда важно, *где* нашлось.

In [ ]:
for found in re.finditer(r"(\w+)=([\d.]+)", metrics):
    print(found.group(1), found.start(), found.end())  # имя поля и его границы в строке

`start()` и `end()` — позиции совпадения в исходной строке; по ним вырезают
контекст вокруг находки или подсвечивают её. Ещё одна функция того же ряда —
`re.split`: это `str.split`, у которого разделитель задан шаблоном.

In [ ]:
print(re.split(r"\s+", " epoch=1   loss=0.7100"))  # ведущий пробел даст пустой кусок
print(" epoch=1   loss=0.7100".split())            # а str.split() его выбросит
print(re.split(r"[= ]", "epoch=1 loss=0.7100"))    # сразу два разных разделителя

`re.split` — не полный синоним `str.split()`: по краям он оставляет пустые куски
(ведущий пробел дал `''` в начале списка), а `str.split()` без аргументов их
отбрасывает. Зато шаблон умеет то, чего `str.split` не может: третий вызов режет
сразу по двум разным разделителям.

Следующая ловушка — **жадность** квантификаторов.

In [ ]:
markup = "<b>loss</b> <i>acc</i>"
print(re.findall(r"<.+>", markup))   # жадный +: одно совпадение от первого < до последнего >
print(re.findall(r"<.+?>", markup))  # ленивый +?: каждый тег отдельно

#### ❓ **Вопрос**: Почему `<.+>` вернул одно длинное совпадение, а `<.+?>` — четыре коротких?

<details>

<summary><strong>Ответ</strong></summary>

`+` жадный: он забирает максимум символов, при котором остаток шаблона всё ещё совпадает. `.` подходит и к `>`, поэтому совпадение растянулось от первого `<` до последнего `>`. Знак `?` после квантификатора делает его ленивым — берётся минимум, и каждый тег совпадает отдельно.

</details>

Регулярка умеет не только искать, но и заменять: `re.sub` подставляет вместо
совпадения новый текст, а `\1` в замене ссылается на группу.

In [ ]:
tail = "run finished user=ivanov token=s3cr3t-9f2a"
print(re.sub(r"token=\S+", "token=***", tail))    # затираем секрет, имя поля оставляем
print(re.sub(r"user=(\w+)", r"user=<\1>", tail))  # \1 в замене — текст первой группы

Поведение шаблона меняют **флаги**. Самый нужный при разборе логов —
`re.MULTILINE`: без него `^` и `$` — это начало и конец всего переданного текста,
а с ним — начало и конец каждой строки. Это важно, когда файл читают целиком, а
не построчно.

In [ ]:
log_text = "2026-11-03 INFO epoch=1\n2026-11-03 ERROR failed\n"
print(re.findall(r"^\S+ (\w+)", log_text))                 # ^ = начало всего текста
print(re.findall(r"^\S+ (\w+)", log_text, re.MULTILINE))   # ^ = начало каждой строки

#### ❓ **Вопрос**: Первый вызов нашёл только `INFO`, хотя строк в тексте две. Почему, и что изменил флаг?

<details>

<summary><strong>Ответ</strong></summary>

Без флагов `^` привязан к началу всего текста, поэтому совпадение возможно ровно одно — в самом начале, и это первая строка с `INFO`. `re.MULTILINE` заставляет `^` совпадать после каждого перевода строки, поэтому вторая строка с `ERROR` тоже попадает в результат.

Альтернатива — читать файл построчно через `splitlines()` и применять шаблон к каждой строке отдельно: тогда флаг не нужен.

</details>

Второй по частоте флаг — `re.IGNORECASE`. В логах разных систем уровень пишут и
`ERROR`, и `Error`, и `error`, и шаблон не должен от этого зависеть.

In [ ]:
print(re.findall(r"error", log_text))                  # регистр важен: ничего не нашлось
print(re.findall(r"error", log_text, re.IGNORECASE))   # а так нашёлся ERROR

Длинный шаблон нечитаем. Флаг `re.VERBOSE` разрешает переносы и комментарии
внутри шаблона — пробелы при этом игнорируются, а нужный пробел пишут как `\s`.

In [ ]:
row_re = re.compile(r"""
    epoch=(?P<epoch>\d+)      # номер эпохи
    \s+                       # один и больше пробелов между полями
    loss=(?P<loss>[\d.]+)     # значение функции потерь: цифры и точка
""", re.VERBOSE)

Шаблон готов — применим его к знакомой строке.

In [ ]:
print(row_re.search(line).groupdict())  # тот же результат, что у шаблона в одну строку

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Регулярные выражения старше компьютеров, на которых их обычно показывают: их
описал математик Стивен Клини в 1951 году как способ задать множество строк. В
инструмент их превратил всё тот же Кен Томпсон, вкрутив в редактор `ed`, — и
команда `g/re/p` («globally find re and print») дала имя утилите grep.

Обратная сторона: модуль `re`, как и большинство реализаций, ищет с возвратами
(backtracking), и на неудачном шаблоне вроде `(a+)+$` время растёт экспоненциально
от длины строки. Именно так в июле 2019-го легла половина Cloudflare: одна
регулярка в фильтре запросов съела процессор на всех серверах разом. На учебных
логах это не грозит, но знать про такую возможность полезно.

</details>

## 5. Собираем парсер лога

Соберём инструменты вместе — ровно так устроена домашка. Записываем учебный лог:
в нём есть и содержательные строки, и посторонние.

In [ ]:
log_path = work / "train.log"
log_path.write_text("""2026-11-03 12:04:12 INFO epoch=1 loss=0.7100
2026-11-03 12:08:45 WARNING nan detected in batch 17
2026-11-03 12:08:46 INFO epoch=2 loss=0.4213
""", encoding="utf-8")
print("лог записан:", log_path.name)  # вторая строка лога — та самая посторонняя

`splitlines()` разрезает текст на список строк по переводам строки и сами `\n`
в результат не кладёт.

In [ ]:
print(log_path.read_text(encoding="utf-8").splitlines()[1])  # вторая строка лога

Теперь разбор: идём по строкам и пробуем шаблон на каждой.

In [ ]:
rows = []
skipped = 0
for raw in log_path.read_text(encoding="utf-8").splitlines():
    found = row_re.search(raw)
    if found is None:      # строка под шаблон не подошла
        skipped += 1       # считаем её, а не выбрасываем молча
        continue
    rows.append((int(found["epoch"]), float(found["loss"])))  # сразу в числа

Смотрим, что получилось и сколько потеряли.

In [ ]:
print(rows)                            # список пар (эпоха, loss)
print("нераспознанных строк:", skipped)  # ненулевое значение — повод посмотреть глазами

Строгую строку из раздела 2 разобрали бы `split` и `partition` (так и сделано в
B4). Здесь они не годятся: в логе есть посторонние строки, и нужен не просто
разрез, а **проверка формата** — совпало или нет.

Схема разбора всегда одна: читаем построчно → пробуем шаблон → совпало
берём группы, не совпало считаем пропуск. `found["epoch"]` — короткая запись для
`found.group("epoch")`.

Считать пропущенные строки обязательно: иначе парсер молча теряет данные, и об
этом узнают уже по странному графику.

И пропущенное — не всегда мусор. Строки traceback принадлежат предыдущей записи:
начало новой видно по дате в начале строки (`re.match(r"\d{4}-\d{2}-\d{2}", raw)`),
а всё остальное приклеивается к последней записи. Ровно это и делает задача H2.

`read_text().splitlines()` выше загрузило **весь файл в память** и сделало из него
список строк. Для учебного лога это нормально, а для лога на сотни мегабайт —
нет: файловый объект сам по себе итератор по строкам, и память не растёт с
размером файла. Открывать его при этом надо через `with` — он закроет файл и при
обычном выходе, и при исключении внутри цикла.

In [ ]:
best = None
with open(log_path, encoding="utf-8") as handle:  # with закроет файл в любом случае
    for raw in handle:                            # итерируем по строкам, не читая файл целиком
        found = row_re.search(raw)
        if found and (best is None or float(found["loss"]) < best):
            best = float(found["loss"])           # запомнили новый минимум
print("лучший loss:", best)

Третья часть — поиск файлов. В домашке лог не один: их находят маской из
раздела 3, а разбор каждого остаётся тем же самым. Разложим наш лог по двум
запускам.

In [ ]:
for name in ["run-01", "run-02"]:
    (runs / name / "train.log").write_text(         # тот же текст в двух запусках
        log_path.read_text(encoding="utf-8"), encoding="utf-8")

И пройдёмся по дереву маской, считая распознанные строки в каждом файле.

In [ ]:
for path in sorted(runs.rglob("train.log")):  # маска ищет файлы, шаблон разбирает содержимое
    text = path.read_text(encoding="utf-8")
    print(path.parent.name, len(row_re.findall(text)))  # имя запуска и число строк с epoch/loss

#### ❓ **Вопрос**: В лог добавилась строка `epoch=6 loss=nan`. Что сделает наш парсер и чем это опасно?

<details>

<summary><strong>Ответ</strong></summary>

Шаблон `[\d.]+` описывает только цифры и точку, а `nan` из букв — совпадения не будет, строка попадёт в `skipped`. Данные не испортятся, но эпоха тихо исчезнет из результата. Именно поэтому счётчик пропусков выводят рядом с результатом: ненулевое значение — сигнал посмотреть, какие строки не разобрались.

</details>

## Итог

- Текст в файле — байты; кодировка — правило перевода. `encoding` указываем явно,
  BOM снимаем через `utf-8-sig`.
- Методы `str` разбирают строгий формат: `strip`, `split`, `partition`,
  `removeprefix`/`removesuffix`. `strip` работает с набором символов, а не с
  подстрокой.
- `glob` и `Path.rglob` собирают файлы по маске; результат сортируем.
- `re` описывает изменчивый формат: группы (лучше именованные), `search` против
  `match`, жадность, `sub` для замены, `re.split` вместо `str.split`, когда
  разделитель задан шаблоном, `finditer` когда нужны позиции, флаги
  `MULTILINE`/`IGNORECASE`/`VERBOSE`.

Домашка — парсер текстового лога на регулярных выражениях. Задачи семинара — в
`tasks.md`, данные готовит `python3 assets/make_data.py`.